# Francisco's `p_storage` optimiser — NPV curve explorer

This notebook loads the same data as `run_hess.py`, then sweeps over 50 candidate
threshold values (5th → 95th percentile of electricity prices) and plots the resulting NPV.

The peak of the curve is the `p_storage` the optimiser picks — and you can see exactly
how far below (or above) the pure breakeven price it lands.

In [1]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# Make 'common' and 'HESS' importable (same as run_hess.py does)
MODEL_ROOT = Path().resolve().parent  # …/2.5. cisco's model/
if str(MODEL_ROOT) not in sys.path:
    sys.path.insert(0, str(MODEL_ROOT))

from common.loaders import build_simulation_df
from HESS.config import HESSConfig
from HESS.dispatch import simulate_hess, simulate_base_case
from HESS.economics import calc_npv_incremental

print('Imports OK')

Imports OK


In [2]:
# ── Load data (same settings as run_hess.py) ──────────────────────────────
BASE_YEARS = [2023, 2024]
P_H2       = 8.0    # €/kg  ← change this to see sensitivity
VT_HOURS   = 264.0  # full-load hours of tank storage

df = build_simulation_df(years=BASE_YEARS, n_project_years=25, farm_capacity_mw=1000.0)
wind_mw = df['wind_mw'].values
prices  = df['price'].values

print(f'Loaded {len(df):,} hourly timesteps')
print(f'Wind : mean {np.nanmean(wind_mw):.1f} MW')
print(f'Price: mean {np.nanmean(prices):.2f} €/MWh')

Loaded 219,000 hourly timesteps
Wind : mean 373.0 MW
Price: mean 76.35 €/MWh


In [ ]:
# ── Build HESS config ─────────────────────────────────────────────────────
el_mw = 350.0
efficiency = 51.5  # kWh/kg

cfg = HESSConfig(
    farm_capacity_mw=1000.0,
    p_el_mw=el_mw,
    e_pem_kwh_per_kg=efficiency,
    vt_hours=VT_HOURS,
    withdraw_rate_daily=0.05,
    p_h2_eur_per_kg=P_H2,
    p_storage=144.23,   # thesis default (overridden in sweep)
)

# Derived values from config
# withdraw_rate_daily is annualized as fraction/day in the model docs,
# so daily off-take equivalent = rate * tank / 365
tank_kg = cfg.cap_h2_tanks_kg
daily_offtake_kg = cfg.withdraw_rate_daily * tank_kg / 365.0

# Pure breakeven: 1 MWh -> (1000/eff) kg -> × price_per_kg
breakeven = (1000.0 / efficiency) * P_H2
print(f'Tank capacity   : {tank_kg:,.0f} kg  ({VT_HOURS:.0f} full-load hours)')
print(f'Daily offtake   : {daily_offtake_kg:,.0f} kg/day')
print(f'Pure breakeven  : EUR {breakeven:.2f}/MWh')
print(f'Thesis p_storage: EUR 144.23/MWh  (Delta = {144.23 - breakeven:+.2f} EUR/MWh vs breakeven)')

TypeError: HESSConfig.__init__() got an unexpected keyword argument 'electrolyser_mw'

In [ ]:
# ── Sweep 50 candidate thresholds and record NPV at each ──────────────────
# (This replicates what optimise_p_storage() does internally)

candidates = np.percentile(prices[prices > 0], np.linspace(5, 95, 50))
base = simulate_base_case(wind_mw, prices, cfg)

npv_values = []
for ps in candidates:
    res = simulate_hess(wind_mw, prices, cfg, p_storage=ps)
    npv = calc_npv_incremental(res, base, cfg)
    npv_values.append(npv)

npv_values = np.array(npv_values)
best_idx = np.argmax(npv_values)
best_ps  = candidates[best_idx]
best_npv = npv_values[best_idx]

print(f'Best p_storage : €{best_ps:.2f}/MWh  (NPV = €{best_npv/1e6:.2f}M)')
print(f'Pure breakeven : €{breakeven:.2f}/MWh')
print(f'Nudge          : {best_ps - breakeven:+.2f} €/MWh vs breakeven')

In [ ]:
# ── Plot NPV vs threshold ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(candidates, npv_values / 1e6, color='steelblue', linewidth=2, marker='o', markersize=4)
ax.axvline(best_ps,    color='limegreen', linestyle='--', linewidth=1.5,
           label=f'Optimal  p_storage = €{best_ps:.1f}/MWh  (NPV max)')
ax.axvline(breakeven,  color='tomato',    linestyle='--', linewidth=1.5,
           label=f'Pure breakeven = €{breakeven:.1f}/MWh  (at €{P_H2}/kg, {efficiency} kWh/kg)')
ax.axvline(144.23,     color='gold',      linestyle=':',  linewidth=1.5,
           label='Thesis value = €144.23/MWh')

ax.set_xlabel('p_storage threshold [€/MWh]')
ax.set_ylabel('Incremental NPV [M€]')
ax.set_title(f'NPV vs Dispatch Threshold  (p_H₂ = €{P_H2}/kg, VT = {VT_HOURS:.0f} h)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

## What the plot tells you

- The **green dashed line** is the threshold the optimiser actually picks — the peak NPV.
- The **red dashed line** is the pure instantaneous breakeven (where grid and H₂ revenue are exactly equal per MWh).
- The **gold dotted line** is the hardcoded thesis value (144.23 €/MWh).

If the green line is **to the left of red** (lower threshold): the optimiser prefers to make H₂ *even slightly above breakeven* to keep the tank primed for the offtake contract — avoiding missed sales later outweighs the per-MWh loss now.

Try changing `P_H2` in cell 2 and re-running — the breakeven shifts and so does the optimal threshold.